### Generación grounded

Este cuaderno integra baseline descriptivo, evidencia estructural, recuperación y salida textual controlada.

#### Atención visual-semántica

En esta versión pequeña no redistribuimos imágenes patrimoniales, por lo que la atención se aborda como **selección de evidencia relevante**: qué campos, qué vecinos y qué restricciones sostienen cada frase.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io_utils import jsonl_load, jsonl_dump

from src.retrieval import fit_text_retriever
from src.evidence_bundle import build_evidence_bundle
from src.generation import grounded_prompt, grounded_generate_template

records = jsonl_load(PROJECT_ROOT / "data_processed/records_master.jsonl")
retriever = fit_text_retriever(records)
record = records[0]
bundle = build_evidence_bundle(record, retriever, top_k=2)
bundle


{'id': 'okr_kh_0068_view_01',
 'title': 'Khipu KH0068 (UR1057, AS057)',
 'object_type': 'Khipu',
 'caption_base': 'Khipu con 589 cordeles, 11 colores registrados y procedencia Unknown.',
 'x_s': {'material': 'fibra textil',
  'cord_count': 589,
  'pendant_count': 558,
  'top_cord': None,
  'knot_types': [],
  'spin_ply': None,
  'color_terms': [],
  'hierarchy': '94 grupos',
  'attachment_structure': None,
  'unique_colors': 11},
 'x_t': {'title': 'Khipu KH0068 (UR1057, AS057)',
  'description': 'Museo Regional de Ica; 94 grupos y 589 cordeles.',
  'keywords': ['Museo Regional de Ica', '589 cords', '11 colors']},
 'x_c': {'chronology': None,
  'geography': 'Ica, Peru',
  'museum': "Museo Regional de Ica 'Adolfo Bermúdez Jenkins'",
  'museum_number': '7',
  'region': 'Unknown',
  'reference_url': 'https://www.khipufieldguide.com/catalog/KH0068.html',
  'dataset_note': 'Curated from Khipu Field Guide public catalog'},
 'plausibility': {'plausibility_score': 0.75, 'plausibility_label': 'a

In [2]:
print(grounded_prompt(record, retriever, top_k=2))


Usa solo la evidencia disponible.
No inventes cronología, procedencia, materialidad ni significado.
Separa observación, metadatos e inferencia.
Explicita incertidumbre.

[OBJETO]
id: okr_kh_0068_view_01
title: Khipu KH0068 (UR1057, AS057)
object_type: Khipu

[BASELINE]
Khipu con 589 cordeles, 11 colores registrados y procedencia Unknown.

[ESTRUCTURA]
{"material": "fibra textil", "cord_count": 589, "pendant_count": 558, "top_cord": null, "knot_types": [], "spin_ply": null, "color_terms": [], "hierarchy": "94 grupos", "attachment_structure": null, "unique_colors": 11}

[TEXTO]
{"title": "Khipu KH0068 (UR1057, AS057)", "description": "Museo Regional de Ica; 94 grupos y 589 cordeles.", "keywords": ["Museo Regional de Ica", "589 cords", "11 colors"]}

[CONTEXTO]
{"chronology": null, "geography": "Ica, Peru", "museum": "Museo Regional de Ica 'Adolfo Bermúdez Jenkins'", "museum_number": "7", "region": "Unknown", "reference_url": "https://www.khipufieldguide.com/catalog/KH0068.html", "dataset

In [3]:
output = grounded_generate_template(record, retriever, top_k=2)
output


{'caption_factual': 'Khipu con 589 cordeles, 11 colores registrados y procedencia Unknown.',
 'nota_tecnico_curatorial': 'Khipu KH0068 (UR1057, AS057) se documenta como khipu asociado a Andean. La ficha disponible sitúa la procedencia en Unknown y aporta como soporte material fibra textil anudada. Los atributos estructurales disponibles incluyen patrón=None, material=fibra textil, análisis=None, jerarquía=94 grupos, cordeles=589.',
 'nota_comparativa': 'La recuperación sugiere afinidad parcial con Khipu KH0328 (UR092); Khipu KH0323 (UR087). Esa proximidad es útil para comparación curatorial, pero no demuestra identidad de origen, función o cronología.',
 'incertidumbre': 'La evidencia actual no permite afirmar una lectura funcional o simbólica cerrada, y cualquier interpretación debe mantenerse provisional.',
 'traza_evidencia': ['baseline: Khipu con 589 cordeles, 11 colores registrados y procedencia Unknown.',
  'provenance: Unknown',
  'material: fibra textil anudada',
  'plausibilit

#### Ruta opcional con un modelo real de captioning

La siguiente celda está desactivada por defecto. Requiere una imagen local real y dependencias de `transformers` y `torch`. La sugerencia es usarla solo como extensión con GPU.

In [4]:
RUN_REAL_CAPTIONING = False
LOCAL_IMAGE_PATH = PROJECT_ROOT / "data_raw" / "demo_real_image.jpg"

if RUN_REAL_CAPTIONING:
    from PIL import Image
    import torch
    from transformers import BlipProcessor, BlipForConditionalGeneration

    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    image = Image.open(LOCAL_IMAGE_PATH).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=40)
    print(processor.decode(out[0], skip_special_tokens=True))
else:
    print("La opción de subtitulado real está desactivada..")


La opción de subtitulado real está desactivada..
